# Whisper large-v3-turbo — DIMER LoRA fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/whisper-asr-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/whisper-asr-pipeline/blob/main/tutorials/whisper_asr_finetune_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-openai%2Fwhisper--large--v3--turbo-ffcc4d?style=flat)](https://huggingface.co/openai/whisper-large-v3-turbo) [![Upstream](https://img.shields.io/badge/Upstream-openai%2Fwhisper-181717?style=flat&logo=github&logoColor=white)](https://github.com/openai/whisper) [![arXiv](https://img.shields.io/badge/arXiv-2212.04356-b31b1b.svg)](https://arxiv.org/abs/2212.04356)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** parameter-efficient (LoRA) domain adaptation of the pinned `openai/whisper-large-v3-turbo` weights for automatic speech recognition, evaluated by corpus word error rate before and after adaptation and after a fresh reload of the exported adapter bundle

**This notebook is standalone.** It carries the repository's pipeline module (`src/whisper_asr_pipeline/pipeline.py` at revision `5cff0a87c84f`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `41f01f3fe87f28c78e2fbf8b568835947dd65ed9` (~1622 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** This notebook needs a 16 GiB-class CUDA GPU (Colab: Runtime > Change runtime type > T4 GPU; Kaggle P100/T4 also works) — the default configuration was measured at 4.5 GiB peak GPU memory and about 27 minutes on a Kaggle P100 (RUN11/RUN12; on CPU it runs but takes hours, so reduce `TRAIN_CLIPS`, `EVAL_CLIPS` and `EPOCHS` first). Once that runtime is selected, **Run all** installs the pinned dependencies, stages and digest-verifies the pinned snapshot, downloads one locale of the public `PolyAI/minds14` corpus at a pinned dataset revision and decodes/resamples it with the pinned `soundfile`/`torchaudio`, draws a seeded train/held-out split and validates every clip into an input manifest, records the zero-shot **baseline** corpus WER through the carried pipeline, attaches LoRA adapters to the verified base weights and trains them for two bounded epochs with AdamW under mixed precision, evaluates the adapted model on the held-out split through the same pipeline and writes the evaluation report, exports the manifested adapter bundle, and reloads it against the verified base snapshot with a weight-level merge check and a transcript agreement check. No repository clone, DIMER worker or service, credential, upload dialog or configuration edit is required (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to upload a `transcripts.csv` (`file,text` columns) plus the referenced audio files; they enter the same decoding, validation, seeded split, baseline, LoRA training, evaluation, export and fresh-reload cells as the public sample (DAT14). Expected format, the 30-second clip ceiling and the privacy guidance are stated in the Prerequisites and in Section 4; uploads stay inside this runtime. BYOD is optional and never part of the default path.

**What is trained and what is not.** LoRA keeps every original weight matrix $W_0$ frozen and learns a low-rank correction $W = W_0 + \frac{\alpha}{r} B A$; $B$ starts at zero, so at step 0 the adapted model *is* the base model. Adapters go on the query and value projections of every attention block in both the 32-layer encoder and the 4-layer decoder (the standard Whisper recipe); with rank 32 that is under 1% of the 809M parameters. What the upstream checkpoint supplies is the weights, tokenizer and feature-extractor configuration; what the carried pipeline module adds is snapshot verification, the input contract, the `corpus_word_error_rate`, `validate_inputs`, `export_adapter_bundle`, `verify_adapter_merge` and `adaptation_report` helpers, and the single decoding path (`WhisperASRPipeline`) that the baseline, the in-memory adapted model and the reloaded bundle all go through, so their transcripts are comparable. **Training loss and validation loss are optimisation evidence only** (FT7); the metric that matters is the corpus WER on the held-out split, and it is tutorial evidence for one small split of one locale, not a benchmark.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision, load a public labelled speech corpus at a pinned revision (or upload your own) and validate it into an input manifest, record a zero-shot baseline through the pipeline, attach LoRA adapters and train them with a plain PyTorch loop whose every knob is explicit, evaluate the adapted model on the held-out split and read the evaluation report correctly, export a manifested adapter bundle, prove that a fresh load applies the saved adapter to the base weights, and export machine-readable outputs plus provenance.

**This notebook does not demonstrate:** full-parameter fine-tuning, encoder-only or decoder-only adaptation, learning-rate schedules, speaker diarization or identification, character-level metrics for languages without whitespace-delimited words (`zh-CN`, `ko-KR`), a forgetting check outside the adaptation distribution, or any calibrated transcript-confidence threshold. Whisper can hallucinate fluent text on silence or music, and the pipeline does not detect it.

## Prerequisites

- **Runtime:** a fresh supported runtime with a 16 GiB-class CUDA GPU (Google Colab **T4** or Kaggle P100/T4; Python 3.12 on Colab). Measured on a Kaggle P100 with the default configuration by the previous, clone-based revision of this notebook: 4.5 GiB peak GPU memory and about 27 minutes end to end, of which about 18 minutes is training (100 optimizer steps) and 4 minutes is the pinned install. CPU execution is supported only as a reduced smoke test: lower `TRAIN_CLIPS`, `EVAL_CLIPS` and `EPOCHS` first, or the training cell will take hours.
- **Knowledge:** basic Python and PyTorch; what a waveform, a sampling rate, a log-mel spectrogram, teacher forcing and a word error rate are; the idea of a low-rank adapter.
- **Data:** the default sample is one locale of the public `PolyAI/minds14` banking-intent corpus (CC-BY-4.0), fetched from the Hugging Face Hub at the pinned dataset revision `40ce77cb32a384e4d50a568e1ec39ac804019d33` and decoded with the pinned `soundfile`/`torchaudio` — no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: a `transcripts.csv` with `file,text` columns plus the referenced audio files (WAV/FLAC/OGG readable by `soundfile`, any rate, mono or stereo, each at most 30 seconds), all selected in the same upload dialog; set `BYOD_LANGUAGE` to the ISO code Whisper should decode in. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API. An adapter trained on the default sample inherits CC-BY-4.0 attribution obligations.
- **External access:** the Hugging Face Hub only, to fetch the pinned `openai/whisper-large-v3-turbo` snapshot (~1622 MB in total) at revision `41f01f3fe87f…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's tools/finetune-pins.txt at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers`, `peft` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.6.0',
    'torchvision==0.21.0',
    'torchaudio==2.6.0',
    'transformers==4.52.1',
    'accelerate==1.3.0',
    'numpy==1.26.4',
    'soundfile==0.13.1',
    'datasets==4.4.0',
    'pandas==2.2.3',
    'peft==0.17.1',
]
NOTEBOOK_SOURCE = {
    'repository': 'whisper-asr-pipeline',
    'repository_revision': '5cff0a87c84ffb6f9b398d53792283c2d269559c',
    'embedded_module': 'src/whisper_asr_pipeline/pipeline.py',
    'embedded_modules': ['src/whisper_asr_pipeline/pipeline.py'],
    'module_sha256': '1febc8f7f7d0bbffa1fc078e3ab956f2fab102c4357f6587473399085c0b3694',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, peft
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'peft': peft.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/whisper_asr_pipeline/` @ `5cff0a87c84f`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/whisper_asr_pipeline/pipeline.py`

In [ ]:
"""Multilingual speech recognition with the pinned ``openai/whisper-large-v3-turbo`` checkpoint.

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the architecture comes from the pinned ``transformers`` release, the
weights are SafeTensors, and no model-repository code is executed.
"""

from __future__ import annotations

import hashlib
import json
import re
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

MODEL_ID = "openai/whisper-large-v3-turbo"
MODEL_REVISION = "41f01f3fe87f28c78e2fbf8b568835947dd65ed9"
MODEL_LICENSE = "MIT"
MODEL_KEY = "whisper-large-v3-turbo"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHTS_FILE = "model.safetensors"
CONFIG_FILE = "config.json"

TASKS = ("transcribe", "translate")
MIN_CHUNK_LENGTH_S = 1
MAX_CHUNK_LENGTH_S = 30  # Whisper's receptive field; longer audio is chunked by the transformers pipeline
# A LoRA bundle is accepted only in the safetensors format; peft would otherwise fall back to a
# pickle-based adapter_model.bin, which this repository's trust boundary refuses.
ADAPTER_WEIGHTS = "adapter_model.safetensors"
# Basic WER normalization: case-fold and drop punctuation so that "classes," and "gospel."
# match an unpunctuated reference. Curly apostrophes are folded to the straight form first;
# word-internal apostrophes and hyphens are kept (a hyphenated compound stays one token).
# Numbers, abbreviations and spelled-out forms are NOT normalized ("Mr." vs "Mister" is an
# error).
_APOSTROPHES = str.maketrans({"\u2019": "'", "\u2018": "'", "\u02bc": "'"})
_PUNCTUATION = re.compile(r"[^\w\s'-]|(?<!\w)['-]|['-](?!\w)", re.UNICODE)


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path or DEFAULT_WEIGHTS_DIR)
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest and the
    tokenizer/config files but git-ignores the weights). Returns the relative paths fetched;
    `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _tokens(text: str) -> list[str]:
    return _PUNCTUATION.sub(" ", text.casefold().translate(_APOSTROPHES)).split()


def word_error_count(reference: str, hypothesis: str) -> tuple[int, int]:
    """Word-level edit distance and reference length after basic normalization.

    Summing the pairs over a corpus and dividing gives the corpus WER; ``word_error_rate``
    is the single-utterance ratio.
    """
    reference_tokens = _tokens(reference)
    hypothesis_tokens = _tokens(hypothesis)
    if not reference_tokens:
        return len(hypothesis_tokens), 0

    previous = list(range(len(hypothesis_tokens) + 1))
    for row_index, reference_token in enumerate(reference_tokens, 1):
        current = [row_index]
        for column_index, hypothesis_token in enumerate(hypothesis_tokens, 1):
            current.append(
                min(
                    current[-1] + 1,
                    previous[column_index] + 1,
                    previous[column_index - 1] + (reference_token != hypothesis_token),
                )
            )
        previous = current
    return previous[-1], len(reference_tokens)


def word_error_rate(reference: str, hypothesis: str) -> float:
    """Word error rate after basic normalization (lowercase, punctuation removed)."""
    errors, reference_length = word_error_count(reference, hypothesis)
    if reference_length == 0:
        return 0.0 if errors == 0 else 1.0
    return errors / reference_length


def adapter_digest(adapter_dir: str | Path) -> str:
    """SHA-256 of the bundle's ``adapter_model.safetensors``: the adapter's identity."""
    digest = hashlib.sha256()
    with open(Path(adapter_dir) / ADAPTER_WEIGHTS, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _load_base(
    device: str | None,
    weights_dir: str | Path | None,
    allow_download: bool,
    adapter_dir: str | Path | None,
) -> tuple[Any, Any, str, str, Any]:
    """Shared loader: ``(model, processor, source, device, dtype)`` from a digest-verified local
    snapshot or, only when ``allow_download`` is set and no snapshot exists, from the Hub at the
    pinned revision. There is no silent fallback: a missing or unverified snapshot raises unless
    downloading was explicitly allowed (MOD8). ``trust_remote_code`` is always False."""
    # The bundle-shape check runs before the heavyweight imports so a malformed adapter directory is
    # refused (and testable) without torch installed.
    adapter_path = None if adapter_dir is None else Path(adapter_dir)
    if adapter_path is not None:
        for required in ("adapter_config.json", ADAPTER_WEIGHTS):
            if not (adapter_path / required).is_file():
                raise FileNotFoundError(f"{required} not found in {adapter_path}")

    import torch
    from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

    resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
    dtype = torch.float16 if resolved_device.startswith("cuda") else torch.float32
    root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
    if (root / MANIFEST_NAME).is_file():
        stage_missing_files(root, allow_download=allow_download)
        verify_snapshot(root)
        # A directory argument makes transformers read config/tokenizer/weights from it directly
        # (no Hub resolution, no cache lookup).
        location: dict[str, Any] = {"pretrained_model_name_or_path": str(root)}
        source = "local-snapshot"
    elif allow_download:
        location = {"pretrained_model_name_or_path": MODEL_ID, "revision": MODEL_REVISION}
        source = "hf-hub"
    else:
        raise FileNotFoundError(
            f"no verified snapshot at {root} and allow_download=False; "
            f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
        )
    processor = AutoProcessor.from_pretrained(**location, trust_remote_code=False)
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        **location,
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
        trust_remote_code=False,
    )
    if adapter_path is not None:
        try:
            from peft import PeftModel
        except ImportError as exc:  # pragma: no cover - depends on the optional extra
            raise ImportError("loading an adapter requires the 'finetune' extra (peft)") from exc
        model = PeftModel.from_pretrained(model, str(adapter_path), is_trainable=False)
        model = model.merge_and_unload()
    if resolved_device.startswith("cuda"):
        model = model.to(resolved_device)
    return model, processor, source, resolved_device, dtype


def load_model(
    device: str | None = None,
    adapter_dir: str | Path | None = None,
    *,
    weights_dir: str | Path | None = None,
    allow_download: bool = True,
) -> tuple[Any, Any]:
    """Return ``(model, processor)`` for the pinned upstream revision, on ``device``.

    Weights are loaded in float16 on CUDA and float32 on CPU, from the digest-verified snapshot
    under ``weights_dir`` when one exists (the fine-tuning tutorial passes the notebook's staged
    directory) and otherwise from the Hub at the pinned revision. With ``adapter_dir``, a PEFT
    LoRA adapter saved by ``PeftModel.save_pretrained`` is attached and merged into the weights,
    so the result is a plain Whisper model; ``peft`` is only imported on that path, and only a
    safetensors bundle is accepted. Remote model code is always refused.
    """
    model, processor, _source, _device, _dtype = _load_base(device, weights_dir, allow_download, adapter_dir)
    return model, processor


def corpus_word_error_rate(references: Sequence[str], hypotheses: Sequence[str]) -> float:
    """Corpus WER: total word edits over total reference words (``word_error_count`` summed)."""
    if len(references) != len(hypotheses):
        raise ValueError("references and hypotheses must have the same length")
    counts = [
        word_error_count(reference, hypothesis)
        for reference, hypothesis in zip(references, hypotheses, strict=True)
    ]
    total_words = sum(length for _, length in counts)
    if total_words == 0:
        raise ValueError("corpus WER is undefined for empty references")
    return sum(errors for errors, _ in counts) / total_words


ADAPTER_BUNDLE_FORMAT = "peft_adapter"
ADAPTER_BUNDLE_FORMAT_VERSION = 1
ARTIFACT_MANIFEST_NAME = "artifact-manifest.json"


def export_adapter_bundle(
    model: Any, adapter_dir: str | Path, *, metrics: Mapping[str, Any], provenance: Mapping[str, Any]
) -> list[dict[str, Any]]:
    """Write the deployable adapter bundle and return its file manifest.

    The bundle is what ``from_pretrained(adapter_dir=...)`` consumes: ``adapter_config.json`` and
    ``adapter_model.safetensors`` from ``PeftModel.save_pretrained`` (safetensors only), plus
    ``metrics.json``, ``provenance.json`` and ``artifact-manifest.json`` (every file with its size
    and SHA-256). Saved LoRA ``B`` matrices that are all zero are refused: such an adapter is a
    no-op and exporting it would hide a training run that never updated anything.
    """
    from safetensors.torch import load_file

    root = Path(adapter_dir)
    if root.exists():
        for stale in sorted(root.rglob("*"), reverse=True):
            stale.unlink() if stale.is_file() else stale.rmdir()
    root.mkdir(parents=True)
    model.save_pretrained(str(root), safe_serialization=True)
    if not (root / ADAPTER_WEIGHTS).is_file():
        raise RuntimeError(f"{ADAPTER_WEIGHTS} was not written; only safetensors adapters are accepted")
    saved_b = [tensor for name, tensor in load_file(str(root / ADAPTER_WEIGHTS)).items() if "lora_B" in name]
    if not saved_b or max(float(tensor.abs().max()) for tensor in saved_b) == 0:
        raise RuntimeError("saved adapter weights are zero or missing")
    (root / "metrics.json").write_text(json.dumps(dict(metrics), indent=2), encoding="utf-8")
    (root / "provenance.json").write_text(json.dumps(dict(provenance), indent=2), encoding="utf-8")
    manifest = [
        {"path": path.relative_to(root).as_posix(), "bytes": path.stat().st_size, "sha256": _sha256(path)}
        for path in sorted(root.rglob("*"))
        if path.is_file()
    ]
    (root / ARTIFACT_MANIFEST_NAME).write_text(
        json.dumps(
            {
                "format": ADAPTER_BUNDLE_FORMAT,
                "formatVersion": ADAPTER_BUNDLE_FORMAT_VERSION,
                "files": manifest,
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    return manifest


def verify_adapter_merge(
    adapter_dir: str | Path,
    module_name: str,
    base_weight: Any,
    reloaded_weight: Any,
    *,
    rank: int,
    alpha: int,
    tolerance: float,
) -> dict[str, Any]:
    """Weight-level proof that a fresh load applied the saved adapter to the base weights.

    For one probe module, ``W_reloaded`` must equal ``W_base + (alpha / rank) * B @ A`` read back
    from the bundle, within ``tolerance``; an adapter whose delta is below the verification
    resolution is rejected too, because the check would then pass on an unmodified base.
    """
    import torch
    from safetensors.torch import load_file

    saved = load_file(str(Path(adapter_dir) / ADAPTER_WEIGHTS))
    lora_a = saved[f"base_model.model.{module_name}.lora_A.weight"].to(torch.float32)
    lora_b = saved[f"base_model.model.{module_name}.lora_B.weight"].to(torch.float32)
    base = base_weight.detach().to("cpu", torch.float32)
    expected = base + (alpha / rank) * (lora_b @ lora_a)
    reloaded = reloaded_weight.detach().to("cpu", torch.float32)
    adapter_delta = float((expected - base).abs().max())
    merge_error = float((reloaded - expected).abs().max())
    if adapter_delta <= 4 * tolerance:
        raise RuntimeError(
            f"adapter delta {adapter_delta:.2e} on {module_name} is below the {tolerance:.0e} verification "
            "resolution; train longer or with a higher learning rate"
        )
    if merge_error > tolerance:
        raise RuntimeError(
            f"reloaded weights differ from base + scaled B@A by {merge_error:.2e} "
            f"(> {tolerance:.0e}) on {module_name}"
        )
    return {
        "module": module_name,
        "adapter_delta_max": adapter_delta,
        "merge_error_max": merge_error,
        "tolerance": tolerance,
    }


def adaptation_report(
    *,
    baseline_wer: float,
    adapted_wer: float,
    reloaded_wer: float | None,
    n_eval: int,
    history: Sequence[Mapping[str, Any]],
    dataset: Mapping[str, Any],
    sample_kind: str = "public-sample",
) -> dict[str, Any]:
    """Evaluation stage for the fine-tuning tutorial: corpus WER before/after adaptation.

    The verdict is always ``sample-sanity``: one held-out split of one corpus says whether the
    adapter helped *here*, not how it generalises, and ``history`` (teacher-forced losses) is
    optimisation evidence only (FT7).
    """
    metrics = [
        {
            "id": "baseline_wer",
            "value": baseline_wer,
            "estimation": f"corpus WER over {n_eval} held-out utterances, zero-shot",
        },
        {
            "id": "adapted_wer",
            "value": adapted_wer,
            "estimation": f"corpus WER over the same {n_eval} utterances, in-memory adapter",
        },
    ]
    if reloaded_wer is not None:
        metrics.append(
            {
                "id": "reloaded_wer",
                "value": reloaded_wer,
                "estimation": "same split, adapter reloaded from the bundle",
            }
        )
    return {
        "task": "automatic speech recognition (LoRA adaptation)",
        "score_semantics": "generated transcript; the pipeline exposes no confidence score or threshold",
        "sample_kind": sample_kind,
        "n_utterances": n_eval,
        "dataset": dict(dataset),
        "baselines": [
            {"id": "zero_shot_base_model", "word_error_rate": baseline_wer},
        ],
        "metrics": metrics,
        "optimisation_history": [dict(row) for row in history],
        "verdict": "sample-sanity",
        "reason": (
            "one seeded held-out split of one corpus; a few points of WER can change sign with another seed"
        ),
        "needs": (
            "a referenced evaluation set from the deployment domain (speakers, microphones, noise) of "
            "several "
            "hundred utterances, and a check outside the adaptation distribution for forgetting, before any "
            "generalisable claim"
        ),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "one audio input: a local file path readable by the ASR stack, an http(s) URL, or a dict "
        "{'array': float waveform, 'sampling_rate': int}"
    ),
    "task": list(TASKS),
    "language": "optional ISO language name/code forwarded to Whisper; None lets the model detect it",
    "chunk_length_s": [MIN_CHUNK_LENGTH_S, MAX_CHUNK_LENGTH_S],
    "preprocessing": (
        "the transformers ASR pipeline resamples to 16 kHz, computes 128-bin log-Mel features in 30 s "
        "windows and chunks longer audio; nothing is altered by this module"
    ),
}


def _check_inputs(audio: Any, task: str, chunk_length_s: int) -> None:
    """Raise ValueError/FileNotFoundError naming the first violated rule (shared with transcribe)."""
    if task not in TASKS:
        raise ValueError("task must be 'transcribe' or 'translate'")
    if (
        isinstance(audio, str | Path)
        and not str(audio).startswith(("http://", "https://"))
        and not Path(audio).is_file()
    ):
        raise FileNotFoundError(f"audio file not found: {audio}")
    if not MIN_CHUNK_LENGTH_S <= chunk_length_s <= MAX_CHUNK_LENGTH_S:
        raise ValueError(
            f"chunk_length_s must be between {MIN_CHUNK_LENGTH_S} and {MAX_CHUNK_LENGTH_S} seconds"
        )


def _observe(audio: Any) -> dict[str, Any]:
    if isinstance(audio, Mapping):
        array = audio.get("array")
        rate = audio.get("sampling_rate")
        samples = len(array) if array is not None and hasattr(array, "__len__") else None
        has_rate = isinstance(rate, int | float) and bool(rate)
        seconds = round(samples / rate, 3) if samples is not None and has_rate else None
        return {"kind": "waveform", "samples": samples, "sampling_rate": rate, "seconds": seconds}
    if isinstance(audio, str | Path) and str(audio).startswith(("http://", "https://")):
        return {"kind": "url", "value": str(audio)}
    path = Path(audio)
    return {"kind": "file", "name": path.name, "bytes": path.stat().st_size}


def validate_inputs(
    audio: str | Path | Mapping[str, Any],
    *,
    language: str | None = None,
    task: str = "transcribe",
    chunk_length_s: int = 30,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observed input, request, verdict).

    Rejection is reported by raising exactly as ``transcribe`` would; a caller that wants the
    finding recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _check_inputs(audio, task, chunk_length_s)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (one audio input per call)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "audio-0", **_observe(audio)}],
        "task": task,
        "language": language,
        "chunk_length_s": chunk_length_s,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any], reference: str | None = None, *, sample_kind: str = "public-sample"
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With a ``reference`` transcript the report carries ``word_error_rate`` (the repository's own
    normalised WER) as sample-sanity evidence for that one utterance; without one the verdict is
    ``not-measurable`` and the report says what would make the task measurable.
    """
    base = {
        "task": f"automatic speech recognition ({result.get('task', 'transcribe')})",
        "score_semantics": "generated transcript; the pipeline exposes no confidence score or threshold",
        "sample_kind": sample_kind,
        "n_utterances": 1,
        "language": result.get("language"),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if reference is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no reference transcript was supplied for the evaluated audio",
            "needs": (
                "reference transcripts for audio from the deployment domain (speakers, microphones, noise), "
                "scored with word_error_rate after the same normalisation; several hundred utterances before "
                "any rate is quoted"
            ),
        }
    return {
        **base,
        "metrics": [
            {
                "id": "word_error_rate",
                "value": word_error_rate(reference, str(result["text"])),
                "normalisation": "casefold, punctuation removed, curly apostrophes folded",
                "estimation": "single utterance, no dispersion estimate",
            }
        ],
        "verdict": "sample-sanity",
        "reason": "one referenced utterance from the tutorial sample; not a benchmark",
        "needs": "a referenced evaluation set from the deployment domain for any generalisable WER claim",
    }


@dataclass
class WhisperASRPipeline:
    _runner: Callable[..., dict[str, Any]]
    device: str
    adapter: str | None = None
    adapter_sha256: str | None = None
    source: str = "injected"

    @property
    def model(self) -> Any:
        """The underlying Transformers model, for inspection (weights, config, dtype)."""
        return self._runner.model

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        adapter_dir: str | Path | None = None,
    ) -> WhisperASRPipeline:
        """Build the pipeline from a digest-verified local snapshot (or, only when
        ``allow_download`` is set, from the Hub at the pinned revision), optionally merging a
        PEFT LoRA adapter (safetensors only). There is no silent fallback: a missing or
        unverified snapshot raises unless downloading was explicitly allowed."""
        model, processor, source, resolved_device, dtype = _load_base(
            device, weights_dir, allow_download, adapter_dir
        )
        adapter_path = None if adapter_dir is None else Path(adapter_dir)
        return cls.from_model(
            model,
            processor,
            adapter=None if adapter_path is None else str(adapter_path),
            adapter_sha256=None if adapter_path is None else adapter_digest(adapter_path),
            source=source,
            device=resolved_device,
            dtype=dtype,
        )

    @classmethod
    def from_model(
        cls,
        model: Any,
        processor: Any,
        adapter: str | None = None,
        adapter_sha256: str | None = None,
        source: str = "injected",
        device: str | None = None,
        dtype: Any | None = None,
    ) -> WhisperASRPipeline:
        """Wrap an already-loaded Whisper model (plain or PEFT-wrapped) in the same decoding path.

        Lets a caller that holds a live model — the fine-tuning tutorial, between training and
        export — transcribe through exactly the pipeline that ``from_pretrained`` builds, so
        in-memory and reloaded results are comparable.
        """
        from transformers import pipeline

        resolved_device = device or str(model.device)
        runner = pipeline(
            "automatic-speech-recognition",
            model=model,
            tokenizer=processor.tokenizer,
            feature_extractor=processor.feature_extractor,
            torch_dtype=model.dtype if dtype is None else dtype,
            device=resolved_device,
        )
        return cls(runner, resolved_device, adapter, adapter_sha256, source)

    def transcribe(
        self,
        audio: str | Path | dict[str, Any],
        *,
        language: str | None = None,
        task: str = "transcribe",
        return_timestamps: bool = False,
        chunk_length_s: int = 30,
    ) -> dict[str, Any]:
        _check_inputs(audio, task, chunk_length_s)

        generate_kwargs: dict[str, Any] = {"task": task}
        if language:
            generate_kwargs["language"] = language
        raw = self._runner(
            str(audio) if isinstance(audio, Path) else audio,
            return_timestamps=return_timestamps,
            chunk_length_s=chunk_length_s,
            generate_kwargs=generate_kwargs,
        )
        if not isinstance(raw, dict) or "text" not in raw:
            raise RuntimeError("ASR backend returned an invalid result")
        return {
            "text": str(raw["text"]).strip(),
            "chunks": raw.get("chunks"),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "task": task,
            "language": language,
            "device": self.device,
            "adapter": self.adapter,
            "adapter_sha256": self.adapter_sha256,
            "source": self.source,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `12`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `41f01f3fe87f…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `WhisperASRPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "whisper-large-v3-turbo",
  "modelId": "openai/whisper-large-v3-turbo",
  "revision": "41f01f3fe87f28c78e2fbf8b568835947dd65ed9",
  "files": [
    {
      "path": "README.md",
      "bytes": 21196,
      "sha256": "aaef74a740faca90fa1899c4233ebe17f2093b9846d0acf16d9131bf650e9585"
    },
    {
      "path": "added_tokens.json",
      "bytes": 34648,
      "sha256": "3c51f66c4c21f9e126970078f11ae77a78c74aee8df606ee9daba86e467108e0"
    },
    {
      "path": "config.json",
      "bytes": 1256,
      "sha256": "c5b526b3e3cd64cd8940dabb45e8ba726629e22d8ed389c29b552f9140daf04a"
    },
    {
      "path": "generation_config.json",
      "bytes": 3772,
      "sha256": "cce11bfe3aaa6ae9e072ea2637caaec8795e68d9b67e655a5af16ee509681a4c"
    },
    {
      "path": "merges.txt",
      "bytes": 493869,
      "sha256": "2df2990a395e35e8dfbc7511e08c12d56018d8d04691e0133e5d63b21e154dc6"
    },
    {
      "path": "model.safetensors",
      "bytes": 1617824864,
      "sha256": "542566a422ae4f3fd23f1ba11add198fca01bbf82e66e6a2857b3f608b1eb9d1"
    },
    {
      "path": "normalizer.json",
      "bytes": 52666,
      "sha256": "bf1c507dc8724ca9cf9903640dacfb69dae2f00edee4f21ceba106a7392f26dd"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 340,
      "sha256": "7ccc62c6f2765af1f3b46c00c9b5894426835a05021c8b9c01eecb6dfb542711"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 2186,
      "sha256": "baea4ea09372eb4fca86b4e4346139fd73cb807d5087e9de0948e971739c3e74"
    },
    {
      "path": "tokenizer.json",
      "bytes": 2710337,
      "sha256": "297b13372ac43916285644fb9687add3cc62ee2a1adb60da3dc25cc94c1871fd"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 282843,
      "sha256": "844b642c73a91359722f47b35705f7174686df33d252695d8572cf9ac03a6389"
    },
    {
      "path": "vocab.json",
      "bytes": 1036558,
      "sha256": "e2aa043ef015641d363d8288e7c241c85e36a5c761fb303598e0710233344387"
    }
  ],
  "totalBytes": 1622464535
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = WhisperASRPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Load a public labelled speech set or optional BYOD

The default path loads one locale of the public `PolyAI/minds14` banking-intent corpus (CC-BY-4.0) at the pinned dataset revision `40ce77cb32a384e4d50a568e1ec39ac804019d33` so the bytes cannot drift: short telephone-quality utterances stored at **8 kHz**, each with a cased, punctuated transcription. Whisper expects 16 kHz input, so every clip is decoded with the pinned `soundfile` and resampled with the pinned `torchaudio` — the same resampler the transformers ASR pipeline applies at inference time, so training and inference see identical audio. A seeded shuffle then takes `TRAIN_CLIPS` clips for adaptation and a disjoint `EVAL_CLIPS` for the held-out baseline/adapted comparison (SPL1; random splitting assumes the utterances are independent, SPL3); the split is recorded as a digest in the exported provenance.

The locale list is restricted to languages whose transcripts are whitespace-delimited: the package's word error rate splits on whitespace, so it is not a meaningful metric for the corpus's `zh-CN` and `ko-KR` locales. BYOD is optional and disabled by default; expected input is a `transcripts.csv` with `file,text` columns plus the referenced audio files, all selected in the same upload dialog, and `BYOD_LANGUAGE` names the decoding language. Look for a dictionary naming the corpus, the language, the clip counts, the split digest and the clip-length statistics.

In [ ]:
import csv
import hashlib
import io
import random

import soundfile as sf
import torchaudio
from datasets import Audio, load_dataset

LOCALE = 'en-US'  # @param ['en-US', 'en-GB', 'en-AU', 'de-DE', 'fr-FR', 'es-ES', 'it-IT', 'nl-NL', 'pl-PL', 'pt-PT', 'ru-RU', 'cs-CZ']
TRAIN_CLIPS = 400  # @param {type:"integer"}
EVAL_CLIPS = 100  # @param {type:"integer"}
SEED = 0  # @param {type:"integer"}
USE_BYOD = False  # @param {type:"boolean"}
BYOD_LANGUAGE = 'en'  # @param {type:"string"}
SAMPLE_DATASET = 'PolyAI/minds14'
SAMPLE_DATASET_REVISION = '40ce77cb32a384e4d50a568e1ec39ac804019d33'
TARGET_RATE = 16_000


def decode_clip(raw_bytes):
    # Decode with the pinned soundfile, then resample with the pinned torchaudio: the datasets
    # audio feature would decode through torchcodec/FFmpeg, which this runtime does not pin.
    waveform, rate = sf.read(io.BytesIO(raw_bytes), dtype='float32')
    if waveform.ndim > 1:
        waveform = waveform.mean(axis=1)
    if rate != TARGET_RATE:
        waveform = torchaudio.functional.resample(torch.from_numpy(waveform), rate, TARGET_RATE).numpy()
    return waveform


if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    manifest_name = next(name for name in uploaded if name.lower().endswith('.csv'))
    rows = list(csv.DictReader(io.StringIO(uploaded[manifest_name].decode('utf-8-sig'))))
    clips = [{'id': row['file'], 'audio': decode_clip(uploaded[row['file']]), 'text': row['text'].strip()} for row in rows]
    LANGUAGE = BYOD_LANGUAGE.strip().lower()
    DATASET = {'name': 'BYOD', 'config': manifest_name, 'revision': None, 'license': None}
    sample_kind = 'BYOD'
else:
    ds = load_dataset(SAMPLE_DATASET, LOCALE, split='train', revision=SAMPLE_DATASET_REVISION)
    ds = ds.cast_column('audio', Audio(decode=False))
    clips = [{'id': row['path'], 'audio': decode_clip(row['audio']['bytes']), 'text': row['transcription'].strip()} for row in ds]
    LANGUAGE = LOCALE.split('-')[0]
    DATASET = {'name': SAMPLE_DATASET, 'config': LOCALE, 'revision': SAMPLE_DATASET_REVISION, 'license': 'CC-BY-4.0'}
    sample_kind = 'public-sample'

if TRAIN_CLIPS < 1 or EVAL_CLIPS < 1 or TRAIN_CLIPS + EVAL_CLIPS > len(clips):
    raise ValueError(f'TRAIN_CLIPS + EVAL_CLIPS must fit in the {len(clips)} available clips')
order = list(range(len(clips)))
random.Random(SEED).shuffle(order)
train_clips = [clips[i] for i in order[:TRAIN_CLIPS]]
eval_clips = [clips[i] for i in order[TRAIN_CLIPS:TRAIN_CLIPS + EVAL_CLIPS]]
SPLIT_DIGEST = hashlib.sha256('\n'.join(c['id'] + '\t' + c['text'] for c in train_clips + eval_clips).encode('utf-8')).hexdigest()
seconds = [len(c['audio']) / TARGET_RATE for c in train_clips + eval_clips]
print({'dataset': DATASET, 'language': LANGUAGE, 'train_clips': len(train_clips), 'eval_clips': len(eval_clips), 'split_digest': SPLIT_DIGEST[:16], 'seconds_mean': round(sum(seconds) / len(seconds), 2), 'seconds_max': round(max(seconds), 2)})
print({'example_text': train_clips[0]['text']})

## 5. Validate every clip → input manifest

`validate_inputs` is the pipeline's public validation stage and is applied to **every** train and held-out clip before any model runs (VAL1): each waveform must satisfy the same contract `transcribe` enforces (`TASKS`, the `MIN_CHUNK_LENGTH_S`–`MAX_CHUNK_LENGTH_S` window), and this notebook adds the two rules the training loop needs — a non-empty transcript, and a clip no longer than `MAX_CHUNK_LENGTH_S` seconds, because Whisper's fixed 30-second window would silently truncate a longer clip's supervision (VAL6/VAL7: nothing is truncated; an offending clip is a rejection finding and stops the run). The per-clip manifests are folded into one input manifest written to `outputs/whisper_asr_finetune_input_manifest.json`. To show what rejection looks like, the cell also validates a request that breaks the chunk ceiling and records the pipeline's own error message as a finding.

In [ ]:
import json

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'TASKS': list(TASKS), 'MIN_CHUNK_LENGTH_S': MIN_CHUNK_LENGTH_S, 'MAX_CHUNK_LENGTH_S': MAX_CHUNK_LENGTH_S, 'TARGET_RATE': TARGET_RATE}})
input_manifest = {'schema': dict(INPUT_SCHEMA), 'inputs': [], 'task': 'transcribe', 'language': LANGUAGE, 'chunk_length_s': MAX_CHUNK_LENGTH_S, 'split': {'train': len(train_clips), 'eval': len(eval_clips), 'seed': SEED, 'digest': SPLIT_DIGEST}, 'verdict': 'accepted', 'findings': [], 'model_id': MODEL_ID, 'model_revision': MODEL_REVISION}
for role, clip_list in (('train', train_clips), ('eval', eval_clips)):
    for clip in clip_list:
        audio_input = {'array': clip['audio'], 'sampling_rate': TARGET_RATE}
        entry = validate_inputs(audio_input, language=LANGUAGE, task='transcribe', names=[clip['id']])['inputs'][0]
        if not clip['text']:
            raise ValueError(f"{clip['id']}: empty transcript; every training and evaluation clip needs a reference")
        if entry['seconds'] > MAX_CHUNK_LENGTH_S:
            raise ValueError(f"{clip['id']}: {entry['seconds']} s exceeds the {MAX_CHUNK_LENGTH_S} s training window; trim it rather than letting the window truncate its supervision")
        input_manifest['inputs'].append({**entry, 'role': role, 'reference_words': len(clip['text'].split())})
# Demonstrate rejection on a request that breaks a ceiling; the finding is recorded, not swallowed.
try:
    validate_inputs({'array': eval_clips[0]['audio'], 'sampling_rate': TARGET_RATE}, chunk_length_s=MAX_CHUNK_LENGTH_S + 1)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'oversized-chunk-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/whisper_asr_finetune_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print({'validated_clips': len(input_manifest['inputs']), 'verdict': input_manifest['verdict'], 'findings': input_manifest['findings']})

## 6. Record the zero-shot baseline

`pipe` was built in Section 3 from the digest-verified snapshot (float16 on CUDA, float32 on CPU). Before anything is trained, the unmodified model transcribes the held-out clips through `WhisperASRPipeline.transcribe`, and `corpus_word_error_rate` (total word edits over total reference words, after the package's case and punctuation normalisation) is recorded as the **baseline** (EVAL10: the naive comparison every adapted number is read against). The pipeline is then released so the training model has the GPU to itself. Look for the baseline WER and three reference/hypothesis pairs.

In [ ]:
import gc
import time


def transcribe_all(asr, clip_list):
    return [asr.transcribe({'array': c['audio'], 'sampling_rate': TARGET_RATE}, language=LANGUAGE, task='transcribe')['text'] for c in clip_list]


eval_references = [c['text'] for c in eval_clips]
DEVICE = pipe.device
started = time.time()
baseline_transcripts = transcribe_all(pipe, eval_clips)
baseline_wer = corpus_word_error_rate(eval_references, baseline_transcripts)
print({'device': DEVICE, 'source': pipe.source, 'baseline_wer': round(baseline_wer, 4), 'seconds': round(time.time() - started, 1)})
for reference, hypothesis in list(zip(eval_references, baseline_transcripts))[:3]:
    print({'reference': reference, 'baseline': hypothesis, 'wer': round(word_error_rate(reference, hypothesis), 3)})
del pipe
gc.collect()
if DEVICE.startswith('cuda'):
    torch.cuda.empty_cache()

## 7. Attach LoRA adapters to the verified base weights

`load_model(weights_dir=WEIGHTS_DIR)` re-reads the **same digest-verified snapshot** Section 3 staged (no second download, no Hub call) and the model is promoted to float32 master weights (mixed precision runs the matmuls in float16); gradient checkpointing is enabled so the encoder's 30-second activations fit a 16 GiB card at batch size 4. The forced decoder prompt is cleared so training and generation both derive the language/task prefix from the labels and the `generate` arguments rather than a config default. One frozen weight (`PROBE_MODULE`) is copied before the adapters are attached so Section 10 can prove, independently of how much the transcripts moved, that a fresh load applies the saved adapter to the base weights. Look for the trainable-parameter count and its share of the total (FT5).

In [ ]:
from peft import LoraConfig, get_peft_model

LORA_RANK = 32  # @param {type:"integer"}
LORA_ALPHA = 64  # @param {type:"integer"}
LORA_TARGETS = ['q_proj', 'v_proj']

base_model, processor = load_model(device=DEVICE, weights_dir=WEIGHTS_DIR, allow_download=False)
base_model = base_model.float()
PROBE_MODULE = 'model.decoder.layers.0.self_attn.q_proj'
probe_base_weight = base_model.get_submodule(PROBE_MODULE).weight.detach().to('cpu', torch.float32).clone()
base_model.config.forced_decoder_ids = None
base_model.generation_config.forced_decoder_ids = None
base_model.config.use_cache = False
base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})

lora_config = LoraConfig(r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias='none', target_modules=LORA_TARGETS)
model = get_peft_model(base_model, lora_config)
trainable_params, all_params = model.get_nb_trainable_parameters()
print({'lora_targets': LORA_TARGETS, 'trainable_parameters': trainable_params, 'all_parameters': all_params, 'trainable_percent': round(100 * trainable_params / all_params, 3), 'dtype': str(next(model.parameters()).dtype)})

## 8. Train

The loop is deliberately plain PyTorch rather than a `Trainer`, so every moving part is visible (FT4/FT6):

- **Features and labels.** Each clip becomes a 128-bin log-mel spectrogram padded to Whisper's fixed 30-second window by the pinned processor. The label sequence is the tokenizer's rendering of the transcript with its language/task prefix, minus the leading start token (the model prepends it when it shifts labels right), padded with `-100` so padding is ignored by the loss.
- **Mixed precision.** On CUDA the forward pass runs under float16 autocast with a gradient scaler; the LoRA weights and optimizer state stay in float32. On CPU everything is float32 (ENV5).
- **AdamW at a constant learning rate** (`1e-3` is the usual LoRA starting point for Whisper) with `GRAD_ACCUM` micro-batches per optimizer step; there is no warmup or decay schedule. `SEED` drives the split and the epoch shuffles; non-deterministic CUDA kernels remain a source of run-to-run variability (ENV7/ENV8).
- **Validation loss** is the teacher-forced cross-entropy on the held-out clips after each epoch. It tracks whether the adapter is still learning; WER, the metric that matters, is measured in the next section (FT7).

In [ ]:
EPOCHS = 2  # @param {type:"integer"}
BATCH_SIZE = 4  # @param {type:"integer"}
GRAD_ACCUM = 2  # @param {type:"integer"}
LEARNING_RATE = 1e-3  # @param {type:"number"}

processor.tokenizer.set_prefix_tokens(language=LANGUAGE, task='transcribe')
decoder_start = model.config.decoder_start_token_id
use_amp = DEVICE.startswith('cuda')


def make_batch(clip_list):
    features = processor.feature_extractor([c['audio'] for c in clip_list], sampling_rate=TARGET_RATE, return_tensors='pt').input_features
    label_rows = []
    for c in clip_list:
        ids = processor.tokenizer(c['text']).input_ids
        if ids and ids[0] == decoder_start:
            ids = ids[1:]
        label_rows.append(ids)
    width = max(len(ids) for ids in label_rows)
    labels = torch.full((len(label_rows), width), -100, dtype=torch.long)
    for row, ids in enumerate(label_rows):
        labels[row, :len(ids)] = torch.tensor(ids)
    return features.to(DEVICE), labels.to(DEVICE)


def batches(clip_list, size):
    for start in range(0, len(clip_list), size):
        yield clip_list[start:start + size]


def evaluate_loss(clip_list):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.inference_mode():
        for batch in batches(clip_list, BATCH_SIZE):
            features, labels = make_batch(batch)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
                loss = model(input_features=features, labels=labels).loss
            count = int((labels != -100).sum())
            total_loss += loss.item() * count
            total_tokens += count
    return total_loss / total_tokens


optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE)
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
shuffler = random.Random(SEED)
if use_amp:
    torch.cuda.reset_peak_memory_stats()
started = time.time()
history = []
OPTIMIZER_STEPS = 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_clips = train_clips[:]
    shuffler.shuffle(epoch_clips)
    optimizer.zero_grad(set_to_none=True)
    epoch_loss, epoch_tokens, pending, epoch_steps = 0.0, 0, 0, 0
    micro_batches = list(batches(epoch_clips, BATCH_SIZE))
    for step, batch in enumerate(micro_batches, start=1):
        features, labels = make_batch(batch)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
            loss = model(input_features=features, labels=labels).loss
        # The window is measured from its own first micro-batch, so the last (possibly short)
        # window is scaled by its true size and still takes its optimizer step.
        window_start = ((step - 1) // GRAD_ACCUM) * GRAD_ACCUM
        window = min(GRAD_ACCUM, len(micro_batches) - window_start)
        scaler.scale(loss / window).backward()
        count = int((labels != -100).sum())
        epoch_loss += loss.item() * count
        epoch_tokens += count
        pending += 1
        if pending == window:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            pending = 0
            epoch_steps += 1
    expected_steps = -(-len(micro_batches) // GRAD_ACCUM)
    if epoch_steps != expected_steps or pending:
        raise RuntimeError(f'epoch {epoch} took {epoch_steps} optimizer steps, expected {expected_steps} ({pending} micro-batches left unstepped)')
    OPTIMIZER_STEPS += epoch_steps
    train_loss = epoch_loss / epoch_tokens
    validation_loss = evaluate_loss(eval_clips)
    history.append({'epoch': epoch, 'train_loss': round(train_loss, 4), 'validation_loss': round(validation_loss, 4), 'optimizer_steps': epoch_steps})
    print(history[-1])
TRAINING_SECONDS = round(time.time() - started, 1)
PEAK_GPU_GIB = round(torch.cuda.max_memory_allocated() / 1024 ** 3, 2) if use_amp else None
print({'training_seconds': TRAINING_SECONDS, 'peak_gpu_gib': PEAK_GPU_GIB, 'optimizer_steps_total': OPTIMIZER_STEPS})

## 9. Evaluate the adapted model → evaluation report

The live LoRA model is wrapped in the same pipeline that produced the baseline (`WhisperASRPipeline.from_model`), so the adapted transcripts come from exactly the decoding path a user of the package gets (5-beam search, the model's 448-token limit) — not from a hand-rolled `generate` call with different settings, which can disagree with the pipeline on near-tie clips (EVAL8). On CUDA the model is first cast to float16, the precision the pipeline serves at. `adaptation_report` then writes `outputs/whisper_asr_finetune_evaluation_report.json`: baseline and adapted corpus WER, the zero-shot baseline as the comparison, the loss history as optimisation evidence, and the verdict `sample-sanity` (EVAL6) — one seeded split of one locale says whether the adapter helped *here*, not how it generalises. A few changed clips are shown side by side.

In [ ]:
model.eval()
model.config.use_cache = True
if use_amp:
    model = model.half()
adapted = WhisperASRPipeline.from_model(model, processor, adapter='in-memory LoRA')
adapted_transcripts = transcribe_all(adapted, eval_clips)
adapted_wer = corpus_word_error_rate(eval_references, adapted_transcripts)
del adapted
report = adaptation_report(baseline_wer=round(baseline_wer, 4), adapted_wer=round(adapted_wer, 4), reloaded_wer=None, n_eval=len(eval_clips), history=history, dataset=DATASET, sample_kind=sample_kind)
with open('outputs/whisper_asr_finetune_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print({'baseline_wer': report['baselines'][0]['word_error_rate'], 'adapted_wer': adapted_wer, 'wer_delta': round(adapted_wer - baseline_wer, 4), 'verdict': report['verdict']})
changed = [(r, b, a) for r, b, a in zip(eval_references, baseline_transcripts, adapted_transcripts) if b != a]
print({'clips_with_changed_transcript': len(changed)})
for reference, before, after in changed[:3]:
    print({'reference': reference, 'baseline': before, 'adapted': after})

## 10. Export the adapter bundle, then prove it reloads against the verified base

The deliverable is the **adapter bundle**, not a copy of the base model (ART1/ART4): `export_adapter_bundle` writes `adapter_model.safetensors` (a few tens of MB) and `adapter_config.json`, `metrics.json`, `provenance.json` (notebook source revision, base model identifier and immutable revision, dataset identity and split digest, hyperparameters, runtime) and an `artifact-manifest.json` listing every file with its size and SHA-256, and refuses an adapter whose saved `B` matrices are all zero. The bundle is zipped under `outputs/`.

Then the notebook proves the bundle is usable from disk (VER1–VER5): the training model is released, `WhisperASRPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, adapter_dir=...)` loads the **verified base snapshot** and merges the saved adapter, `verify_adapter_merge` checks on the probe module that `W_reloaded == W_base + (alpha / r) * B @ A` read back from the bundle (this proves the adapter was applied to the fresh weights however small its effect on the transcripts), and the held-out clips are transcribed again through the same pipeline — the run fails unless the reloaded transcripts agree with the in-memory adapted transcripts on at least 95% of the clips (float16 decoding is not bit-exact across two loads). The reloaded WER is added to the evaluation report, and the result, metrics and provenance are exported.

In [ ]:
import math
import shutil
import zipfile
from pathlib import Path

ADAPTER_DIR = Path('outputs/whisper-asr-lora-adapter')
ARTIFACT_ZIP = Path('outputs/whisper-asr-lora-adapter.zip')
metrics = {'baseline_wer': round(baseline_wer, 4), 'adapted_wer': round(adapted_wer, 4), 'wer_delta': round(adapted_wer - baseline_wer, 4), 'eval_clips': len(eval_clips), 'history': history}
PROVENANCE = {
    'artifactFormat': ADAPTER_BUNDLE_FORMAT,
    'artifactFormatVersion': ADAPTER_BUNDLE_FORMAT_VERSION,
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'baseModel': MODEL_ID,
    'baseModelRevision': MODEL_REVISION,
    'baseModelLicense': MODEL_LICENSE,
    'trustRemoteCode': False,
    'dataset': {**DATASET, 'language': LANGUAGE, 'train_clips': len(train_clips), 'eval_clips': len(eval_clips), 'seed': SEED, 'split_digest': SPLIT_DIGEST},
    'training': {'method': 'lora', 'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'grad_accum': GRAD_ACCUM, 'learning_rate': LEARNING_RATE, 'lora_rank': LORA_RANK, 'lora_alpha': LORA_ALPHA, 'target_modules': LORA_TARGETS, 'mixed_precision': 'float16' if use_amp else None, 'seconds': TRAINING_SECONDS, 'peak_gpu_gib': PEAK_GPU_GIB, 'optimizer_steps': OPTIMIZER_STEPS},
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'peft': peft.__version__, 'device': DEVICE},
}
manifest = export_adapter_bundle(model, ADAPTER_DIR, metrics=metrics, provenance=PROVENANCE)
ARTIFACT_ZIP.unlink(missing_ok=True)
with zipfile.ZipFile(ARTIFACT_ZIP, 'w', zipfile.ZIP_STORED) as archive:
    for p in sorted(ADAPTER_DIR.rglob('*')):
        if p.is_file():
            archive.write(p, p.relative_to(ADAPTER_DIR).as_posix())
zip_sha256 = hashlib.sha256(ARTIFACT_ZIP.read_bytes()).hexdigest()
print({'adapter_dir': str(ADAPTER_DIR), 'files': len(manifest), 'zip_mib': round(ARTIFACT_ZIP.stat().st_size / 1024 ** 2, 1), 'zip_sha256': zip_sha256})

del model, base_model, optimizer, scaler
gc.collect()
if use_amp:
    torch.cuda.empty_cache()
reloaded = WhisperASRPipeline.from_pretrained(device=DEVICE, weights_dir=WEIGHTS_DIR, allow_download=False, adapter_dir=ADAPTER_DIR)
weight_tolerance = 1e-3 if reloaded.model.dtype == torch.float16 else 1e-5
reload_weight_check = verify_adapter_merge(ADAPTER_DIR, PROBE_MODULE, probe_base_weight, reloaded.model.get_submodule(PROBE_MODULE).weight, rank=LORA_RANK, alpha=LORA_ALPHA, tolerance=weight_tolerance)
reloaded_transcripts = transcribe_all(reloaded, eval_clips)
reloaded_wer = corpus_word_error_rate(eval_references, reloaded_transcripts)
agreement = sum(a == r for a, r in zip(adapted_transcripts, reloaded_transcripts))
if agreement < math.ceil(0.95 * len(eval_clips)):
    raise RuntimeError(f'Reloaded adapter reproduces only {agreement}/{len(eval_clips)} in-memory adapted transcripts')
metrics['reloaded_wer'] = round(reloaded_wer, 4)
metrics['reload_agreement'] = f'{agreement}/{len(eval_clips)}'
metrics['clips_changed_vs_baseline'] = sum(b != r for b, r in zip(baseline_transcripts, reloaded_transcripts))
metrics['reload_weight_check'] = reload_weight_check
report = adaptation_report(baseline_wer=metrics['baseline_wer'], adapted_wer=metrics['adapted_wer'], reloaded_wer=metrics['reloaded_wer'], n_eval=len(eval_clips), history=history, dataset=DATASET, sample_kind=sample_kind)
with open('outputs/whisper_asr_finetune_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print({'reloaded_wer': metrics['reloaded_wer'], 'reload_agreement': metrics['reload_agreement'], 'clips_changed_vs_baseline': metrics['clips_changed_vs_baseline'], 'adapter': reloaded.adapter, 'adapter_sha256': reloaded.adapter_sha256, 'source': reloaded.source, 'reload_weight_check': reload_weight_check})

payload = {
    'metrics': metrics,
    'evaluation_report': report,
    'input_manifest': {k: v for k, v in input_manifest.items() if k != 'inputs'},
    'examples': [{'reference': r, 'baseline': b, 'reloaded': a} for r, b, a in list(zip(eval_references, baseline_transcripts, reloaded_transcripts))[:5]],
    'artifact': {'zip': str(ARTIFACT_ZIP), 'sha256': zip_sha256, 'adapter_sha256': reloaded.adapter_sha256, 'files': manifest},
    'provenance': PROVENANCE,
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'peft': peft.__version__, 'device': reloaded.device},
    'sample': {'kind': sample_kind, 'dataset': DATASET, 'language': LANGUAGE, 'split_digest': SPLIT_DIGEST},
}
with open('outputs/whisper_asr_finetune_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
shutil.copy(ADAPTER_DIR / 'metrics.json', 'outputs/whisper_asr_finetune_metrics.json')
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The adapted transcripts are model-generated. The baseline, adapted and reloaded WER figures are corpus-level numbers on one seeded held-out split of one locale of one public corpus (or of the uploaded BYOD set) and must not be generalized to other languages, speakers, domains, or capture conditions; a WER that moved by a few points on 100 short utterances is within the range where a different seed can change the sign, and no dispersion estimate is computed (EVAL5/ENV8). The adapter specializes the model toward the training distribution and can degrade it elsewhere (catastrophic forgetting); the tutorial measures nothing outside the held-out split. Training and validation loss are optimisation evidence only. The pipeline provides no diarization, speaker identity, biometric inference, or calibrated transcript-confidence threshold, and the adapter inherits the license obligations of both the base weights (MIT) and the training data (CC-BY-4.0 for the default sample).

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, load and resample the demonstrated data, validate it, record a zero-shot baseline through the public pipeline, train LoRA adapters with the shown configuration, export a manifested adapter bundle, and reload that bundle against the verified base with a weight-level merge check and transcript agreement — in the tested runtime, without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence use, or that the adapter generalises beyond the split it was measured on.

**Try next:** change `LOCALE` to another whitespace-delimited language and compare the baseline/adapted delta; halve `LEARNING_RATE` or set `EPOCHS = 1` and watch whether the validation loss and the WER move together; upload a few minutes of your own domain speech through BYOD and read the report's `needs` field before trusting the number.

## References

- Repository README: https://github.com/kurtvalcorza/whisper-asr-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/whisper-asr-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/whisper-asr-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/openai/whisper-large-v3-turbo
- Upstream code: https://github.com/openai/whisper
- Whisper paper: https://arxiv.org/abs/2212.04356
- LoRA paper: https://arxiv.org/abs/2106.09685
- Public sample: https://huggingface.co/datasets/PolyAI/minds14